## Problem Statement

Goal: Predict whether a customer will churn (Yes/No) using historical customer data.

Problem Type: Binary Classification.

## Importing the dependenicies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier


## Load the data + Basic Checks

In [ ]:
data = pd.read_csv("dataset/Telco-Customer-Churn.csv")
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
# data["TotalCharges"] = data["TotalCharges"].astype(float)
data[data["TotalCharges"]==' ']["tenure"]

In [ ]:
len(data[data["TotalCharges"]==' '])

In [ ]:
data["TotalCharges"] = data["TotalCharges"].replace({' ': '0.0'})

In [ ]:
data["TotalCharges"] = data["TotalCharges"].astype(float)

In [ ]:
data['Churn'].value_counts()

### Target Distribution

The dataset contains both churn and non-churn customers.

Checking the class distribution helps determine whether the dataset is imbalanced.  
In this case, around **73% of customers did not churn** and **27% churned**.

This indicates a **moderate class imbalance**, meaning that relying only on accuracy could be misleading. Therefore, additional metrics such as **precision, recall, and F1 score** will be used to evaluate model performance.

## EDA

In [ ]:
sns.countplot(x="Churn", data=data, color="#C44E52")
plt.show()

In [ ]:
sns.countplot(x="Contract", hue="Churn", data=data)
plt.show()

In [ ]:
sns.boxplot(x="Churn", y="tenure", data=data, hue="Churn", palette="gist_ncar")
plt.show()

In [ ]:
sns.boxplot(x="Churn", y="MonthlyCharges", data=data, hue="Churn",palette="plasma")
plt.show()

### Key Observations from EDA

1. **Contract Type**
   - Customers with **month-to-month contracts** have significantly higher churn rates.
   - Long-term contracts (one year / two year) show much lower churn.

2. **Tenure**
   - Customers with **low tenure tend to churn more frequently**.
   - Long-term customers are more likely to stay.

3. **Monthly Charges**
   - Customers paying **higher monthly charges show slightly higher churn rates**.

These patterns suggest that **customer commitment (contract length) and early customer lifecycle stages** are strong churn indicators.

## Data Preprocessing

In [ ]:
X = data.drop("Churn", axis=1)
y = data["Churn"].map({"No": 0, "Yes": 1})
X.head()

In [ ]:
X = X.drop(columns="customerID")
X.head()

In [ ]:
X = pd.get_dummies(X, drop_first=True)
X.head()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Data Preprocessing Steps

Several preprocessing steps were applied before training the model:

1. **Target Encoding**
   - The target variable `Churn` was converted from categorical values (`Yes` / `No`) to numeric values (`1` / `0`).

2. **Removing Identifier**
   - `customerID` was removed because identifiers do not provide predictive information.

3. **Categorical Encoding**
   - Categorical features were converted into numeric format using **one-hot encoding** (`pd.get_dummies`).

4. **Feature Scaling**
   - Features were scaled using **StandardScaler** to ensure consistent magnitude across variables.

These steps prepare the dataset for machine learning algorithms.

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

### Train-Test Split

The dataset was split into **training (80%)** and **testing (20%)** sets.

The `stratify` parameter was used to ensure that the **class distribution of churn vs non-churn remains consistent** in both training and testing datasets.

This helps ensure that the model evaluation is reliable and representative of real-world performance.

In [ ]:
model = LogisticRegression(random_state=42, class_weight='balanced')

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)


In [ ]:
print("Train accuracy: ", train_acc)
print("Test accuracy: ", test_acc)

In [ ]:
print("Accuracy: ", accuracy_score(y_test, test_pred))
print("\nclassification_report: ", classification_report(y_test, test_pred))
print("\nConfusion Matrix: ", confusion_matrix(y_test, test_pred))



### Logistic Regression with Class Balancing

To address the class imbalance, the model was trained with:

class_weight = 'balanced'

This forces the algorithm to give **more importance to the minority class (churn customers)**.

Results:

- Recall for churn increased significantly to **0.78**
- Accuracy decreased slightly to **~74%**

Although accuracy dropped, the model now **detects far more churn customers**, which is often more valuable for retention strategies.

## Cross Validation

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10000, class_weight='balanced'))
])

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)


In [ ]:
print("Cross Validation score: ", scores)
print("Cross value means: ", scores.mean())
print("Cross mean std: ", scores.std())


### Cross Validation

To ensure that model performance is reliable and not dependent on a single train-test split, **5-fold cross-validation** was performed.

Results show:

- Mean accuracy ≈ **0.75**
- Standard deviation ≈ **0.007**

The low standard deviation indicates that the model performance is **stable across different data splits**.

## Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

train_pred = rf.predict(X_train)
test_pred = rf.predict(X_test)

train_acc_rf = accuracy_score(y_train, train_pred)
test_acc_rf = accuracy_score(y_test, test_pred)


In [ ]:
print("Train accuracy: ", train_acc_rf)
print("Test accuracy: ", test_acc_rf)
print("Accuracy: ", accuracy_score(y_test, test_pred))
print("\nclassification_report: ", classification_report(y_test, test_pred))
print("\nConfusion Matrix: ", confusion_matrix(y_test, test_pred))


### Random Forest Model

A Random Forest classifier was trained as a more complex model to capture nonlinear relationships.

Results:

- Test accuracy ≈ **0.79**
- Recall for churn ≈ **0.50**

Although the model achieved reasonable accuracy, its ability to detect churn customers was **worse than the balanced logistic regression model**.

Additionally, the very high training accuracy (~99%) suggests **overfitting**, where the model memorizes training data rather than generalizing well.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier())
])

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)



In [ ]:
print("Cross Validation score: ", scores)
print("Cross value means: ", scores.mean())
print("Cross mean std: ", scores.std())


## Comparison Table

| Model             | Accuracy | Recall (Churn) | Precision | F1       |
| ----------------- | -------- | -------------- | --------- | -------- |
| Logistic          | 0.81     | 0.56           | 0.66      | 0.61     |
| Balanced Logistic | 0.74     | **0.78**       | 0.51      | **0.61** |
| Random Forest     | 0.79     | 0.50           | 0.62      | 0.55     |


## Final Conclusion

Three models were evaluated for customer churn prediction:

1. Logistic Regression
2. Random Forest

Although Logistic Regression achieved the highest accuracy, it failed to detect many churn customers.

Balanced Logistic Regression achieved the **highest recall for churn (0.78)**, meaning it successfully identifies most at-risk customers.

For businesses focused on **customer retention**, detecting potential churners is more important than maximizing accuracy.

Therefore, **Balanced Logistic Regression is the most suitable model for this problem**, as it allows companies to identify more customers who are likely to leave and take proactive retention actions.